# Notebook 08: Model Inference & Packaging Verification

**Project:** MachineMind AI  
**Dataset:** NASA IMS Bearing Dataset (Set 2)  
**Task:** Unsupervised Vibration Anomaly Detection  
**Phase:** Phase 10 of 11 — Model Packaging  
**Document Type:** Reproducible Verification & Demonstration Notebook  
**Date:** September 30, 2026  

---

## Executive Summary & Educational Objective

This notebook provides an end-to-end verification and demonstration of the packaged MachineMind AI model artifacts (`models/iforest_pipeline_v1.joblib`, `models/pca_pipeline_v1.joblib`, `models/model_metadata.json`) and the inference engine (`src/inference.py`).

### Key Educational & Engineering Concepts:
1. **Serialization & Fitted State**: Loading pre-trained scalers (`RobustScaler`) and estimators (`IsolationForest`, `PCA`) without retraining.
2. **Train/Serve Consistency**: Ensuring identical feature extraction (28 time-domain features) and feature ordering to prevent train/serve skew.
3. **Metadata-Driven Inference**: Reading exact calibration thresholds ($P_{99}$) and persistence parameter $k=3$ at runtime from `model_metadata.json`.
4. **Stateful Persistence Filtering**: Maintaining $k=3$ consecutive exceedance counts to eliminate transient false alarms (20-minute confirmation delay).
5. **Logical OR System Aggregation**: Triggering system-level alert if **ANY** channel achieves sustained exceedance.
6. **Numerical Reference Consistency**: Verifying that inference engine scores match reference pipeline scores within tolerance ($\le 10^{-5}$).


## 1. Imports and Environment Setup

We import standard libraries, project utilities (`src.data_loading`), and the `AnomalyInferenceEngine` from `src.inference`.


In [1]:
import sys
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Add project root to sys.path if needed
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loading import DEFAULT_RAW_SET2_PATH, get_snapshot_files, load_snapshot
from src.inference import AnomalyInferenceEngine, CHANNELS

print("Environment setup successful.")
print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version:  {np.__version__}")
print(f"pandas Version: {pd.__version__}")
print(f"joblib Version: {joblib.__version__}")


Environment setup successful.
Python Version: 3.14.2
NumPy Version:  2.5.3
pandas Version: 3.0.6
joblib Version: 1.6.0


## 2. Load & Inspect Packaged Metadata

We load `models/model_metadata.json` directly from disk and inspect its authoritative configuration: data provenance, 28 ordered feature names, model hyperparameters, calibration thresholds, and environment versions.


In [2]:
metadata_path = Path("../models/model_metadata.json")
if not metadata_path.exists():
    metadata_path = Path("models/model_metadata.json")

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("=== PACKAGED MODEL METADATA SUMMARY ===")
print(f"Model Name:    {metadata['model_name']}")
print(f"Version:       {metadata['version']}")
print(f"Creation Date: {metadata['creation_date']}")
print(f"Task:          {metadata['task']}")
print("\nData Provenance:")
print(f"  Dataset: {metadata['provenance']['dataset']} ({metadata['provenance']['subset']})")
print(f"  Fit Partition:         {metadata['provenance']['healthy_period_definition']}")
print(f"  Calibration Partition: {metadata['provenance']['calibration_period_definition']}")
print("\nFeature Set:")
print(f"  Feature Type: {metadata['feature_extraction']['feature_set_type']} ({metadata['feature_extraction']['total_features']} total features)")
print(f"  Ordered Features: {metadata['feature_extraction']['ordered_feature_names'][:4]} ... {metadata['feature_extraction']['ordered_feature_names'][-4:]}")

print("\nIsolation Forest P99 Thresholds:")
for ch, val in metadata['models']['iforest']['channel_thresholds_P99'].items():
    print(f"  {ch}: {val:.6f}")

print("\nPCA Reconstruction Error P99 Thresholds:")
for ch, val in metadata['models']['pca']['channel_thresholds_P99'].items():
    print(f"  {ch}: {val:.6f}")


=== PACKAGED MODEL METADATA SUMMARY ===
Model Name:    MachineMind AI Vibration Anomaly Detector
Version:       1.0.0
Creation Date: 2026-09-30 06:00:50 UTC
Task:          Unsupervised Vibration Anomaly Detection

Data Provenance:
  Dataset: NASA IMS Bearing Dataset (Set 2)
  Fit Partition:         Snapshots 0 to 159 (first ~26.7 hours, assumed healthy)
  Calibration Partition: Snapshots 160 to 179 (~3.3 hours)

Feature Set:
  Feature Type: time_domain_28 (28 total features)
  Ordered Features: ['ch1_mean', 'ch1_std', 'ch1_rms', 'ch1_p2p'] ... ['ch4_p2p', 'ch4_skewness', 'ch4_kurtosis', 'ch4_crest_factor']

Isolation Forest P99 Thresholds:
  ch1: 0.513168
  ch2: 0.472510
  ch3: 0.622558
  ch4: 0.506628

PCA Reconstruction Error P99 Thresholds:
  ch1: 0.460780
  ch2: 0.396495
  ch3: 0.194598
  ch4: 0.503693


## 3. Initialize Anomaly Inference Engines

We instantiate `AnomalyInferenceEngine` for both `iforest` and `pca` model architectures using the packaged artifacts in `models/`.


In [3]:
models_dir = Path("../models") if Path("../models").exists() else Path("models")

engine_iforest = AnomalyInferenceEngine(artifacts_dir=models_dir, model_type="iforest", stateful=False)
engine_pca = AnomalyInferenceEngine(artifacts_dir=models_dir, model_type="pca", stateful=False)

print("Inference Engines initialized successfully.")
print(f"Isolation Forest Engine loaded {len(engine_iforest.thresholds)} channel thresholds.")
print(f"PCA Engine loaded {len(engine_pca.thresholds)} channel thresholds.")


Inference Engines initialized successfully.
Isolation Forest Engine loaded 4 channel thresholds.
PCA Engine loaded 4 channel thresholds.


## 4. Stateless Inference Demonstration on Real Snapshots

We evaluate raw 4-channel snapshots (0..4) using `predict_snapshot()` in stateless mode (`stateful=False`).


In [4]:
raw_dir = DEFAULT_RAW_SET2_PATH if DEFAULT_RAW_SET2_PATH.exists() else Path("../data/raw/IMS/2nd_test/2nd_test")
snapshot_files = get_snapshot_files(raw_dir)

print(f"Loaded {len(snapshot_files)} raw snapshot files. Evaluating first 5 snapshots in stateless mode...\n")

for i in range(5):
    fpath = snapshot_files[i]
    df_snap = load_snapshot(fpath)
    ts = fpath.name
    
    res_if = engine_iforest.predict_snapshot(df_snap, timestamp=ts)
    res_pca = engine_pca.predict_snapshot(df_snap, timestamp=ts)
    
    print(f"Snapshot [{i}] ({ts}):")
    print(f"  IForest Scores: ch1={res_if['channel_scores']['ch1']:.4f}, ch2={res_if['channel_scores']['ch2']:.4f}, ch3={res_if['channel_scores']['ch3']:.4f}, ch4={res_if['channel_scores']['ch4']:.4f}")
    print(f"  IForest Flags:  ch1={res_if['channel_raw_flags']['ch1']}, ch2={res_if['channel_raw_flags']['ch2']}, ch3={res_if['channel_raw_flags']['ch3']}, ch4={res_if['channel_raw_flags']['ch4']}")
    print(f"  PCA Scores:     ch1={res_pca['channel_scores']['ch1']:.4f}, ch2={res_pca['channel_scores']['ch2']:.4f}, ch3={res_pca['channel_scores']['ch3']:.4f}, ch4={res_pca['channel_scores']['ch4']:.4f}")
    print(f"  PCA Flags:      ch1={res_pca['channel_raw_flags']['ch1']}, ch2={res_pca['channel_raw_flags']['ch2']}, ch3={res_pca['channel_raw_flags']['ch3']}, ch4={res_pca['channel_raw_flags']['ch4']}\n")


Loaded 984 raw snapshot files. Evaluating first 5 snapshots in stateless mode...

Snapshot [0] (2004.02.12.10.32.39):
  IForest Scores: ch1=0.7562, ch2=0.7748, ch3=0.7382, ch4=0.7007
  IForest Flags:  ch1=True, ch2=True, ch3=True, ch4=True
  PCA Scores:     ch1=0.0390, ch2=0.0334, ch3=0.0537, ch4=0.0725
  PCA Flags:      ch1=False, ch2=False, ch3=False, ch4=False



Snapshot [1] (2004.02.12.10.42.39):
  IForest Scores: ch1=0.5447, ch2=0.5336, ch3=0.5559, ch4=0.4190
  IForest Flags:  ch1=True, ch2=True, ch3=False, ch4=False
  PCA Scores:     ch1=0.5433, ch2=0.1903, ch3=0.1197, ch4=0.0080
  PCA Flags:      ch1=True, ch2=False, ch3=False, ch4=False



Snapshot [2] (2004.02.12.10.52.39):
  IForest Scores: ch1=0.5385, ch2=0.4658, ch3=0.5511, ch4=0.6262
  IForest Flags:  ch1=True, ch2=False, ch3=False, ch4=True
  PCA Scores:     ch1=0.1220, ch2=0.2400, ch3=0.0394, ch4=1.1004
  PCA Flags:      ch1=False, ch2=False, ch3=False, ch4=True

Snapshot [3] (2004.02.12.11.02.39):
  IForest Scores: ch1=0.7167, ch2=0.5446, ch3=0.6110, ch4=0.7775
  IForest Flags:  ch1=True, ch2=True, ch3=False, ch4=True
  PCA Scores:     ch1=0.2734, ch2=0.0333, ch3=0.0380, ch4=0.4622
  PCA Flags:      ch1=False, ch2=False, ch3=False, ch4=False

Snapshot [4] (2004.02.12.11.12.39):
  IForest Scores: ch1=0.4308, ch2=0.3584, ch3=0.4387, ch4=0.4686
  IForest Flags:  ch1=False, ch2=False, ch3=False, ch4=False
  PCA Scores:     ch1=0.1725, ch2=0.0545, ch3=0.0255, ch4=0.0481
  PCA Flags:      ch1=False, ch2=False, ch3=False, ch4=False



## 5. Numerical Reference Consistency Check

We verify that the scores computed by `AnomalyInferenceEngine` from raw snapshot files match reference pipeline scores derived directly from `features_set2.csv` within the required numerical tolerance of $\le 10^{-5}$.


In [5]:
csv_path = Path("../data/processed/features_set2.csv") if Path("../data/processed/features_set2.csv").exists() else Path("data/processed/features_set2.csv")
df_csv = pd.read_csv(csv_path)

max_diff_iforest = 0.0
max_diff_pca = 0.0

for i in range(5):
    fpath = snapshot_files[i]
    df_snap = load_snapshot(fpath)
    
    res_if = engine_iforest.predict_snapshot(df_snap)
    res_pca = engine_pca.predict_snapshot(df_snap)
    
    for ch in CHANNELS:
        ch_stats = ["mean", "std", "rms", "p2p", "skewness", "kurtosis", "crest_factor"]
        ch_cols = [f"{ch}_{s}" for s in ch_stats]
        x_csv = df_csv.iloc[i:i+1][ch_cols].to_numpy(dtype=np.float64)
        
        ref_if_score = float(engine_iforest.artifact["channels"][ch]["pipeline"].compute_anomaly_scores(x_csv)[0])
        ref_pca_score = float(engine_pca.artifact["channels"][ch]["pipeline"].compute_anomaly_scores(x_csv)[0])
        
        diff_if = abs(res_if["channel_scores"][ch] - ref_if_score)
        diff_pca = abs(res_pca["channel_scores"][ch] - ref_pca_score)
        
        max_diff_iforest = max(max_diff_iforest, diff_if)
        max_diff_pca = max(max_diff_pca, diff_pca)

print("=== NUMERICAL REFERENCE CONSISTENCY VERIFICATION ===")
print(f"Max Absolute Score Difference (Isolation Forest): {max_diff_iforest:.2e}")
print(f"Max Absolute Score Difference (PCA Error):        {max_diff_pca:.2e}")
print(f"Target Numerical Tolerance:                       <= 1.00e-05")

assert max_diff_iforest <= 1e-5, f"Isolation Forest score difference {max_diff_iforest} exceeds tolerance!"
assert max_diff_pca <= 1e-5, f"PCA score difference {max_diff_pca} exceeds tolerance!"

print("\nSUCCESS: Numerical reference consistency verified within tolerance!")


=== NUMERICAL REFERENCE CONSISTENCY VERIFICATION ===
Max Absolute Score Difference (Isolation Forest): 0.00e+00
Max Absolute Score Difference (PCA Error):        1.77e-14
Target Numerical Tolerance:                       <= 1.00e-05

SUCCESS: Numerical reference consistency verified within tolerance!


## 6. Stateful $k=3$ Persistence & Reset State Demonstration

We demonstrate stateful inference (`stateful=True`) and verify the exact $k=3$ consecutive exceedance logic:
- Exceedance 1: `count = 1`, `sustained_flag = False`
- Exceedance 2: `count = 2`, `sustained_flag = False`
- Exceedance 3: `count = 3`, `sustained_flag = True` (Sustained Alert Active)
- Exceedance 4: `count = 4`, `sustained_flag = True` (Sustained Alert Retained)
- Non-exceedance: `count = 0`, `sustained_flag = False` (Immediate Reset)


In [6]:
engine_stateful = AnomalyInferenceEngine(artifacts_dir=models_dir, model_type="iforest", stateful=True)
engine_stateful.reset_state()

print("=== STATEFUL k=3 PERSISTENCE DEMONSTRATION ===")

# Force ch1 exceedance by setting threshold low for demonstration
for ch in CHANNELS:
    engine_stateful.thresholds[ch] = 999.0
engine_stateful.thresholds["ch1"] = -999.0  # Force raw exceedance on ch1 only

sample_snap = load_snapshot(snapshot_files[0])

# Sequence of 4 exceedances followed by 1 non-exceedance
steps = ["Exceedance 1", "Exceedance 2", "Exceedance 3", "Exceedance 4", "Non-Exceedance"]

for step_idx, step_name in enumerate(steps):
    if step_name == "Non-Exceedance":
        engine_stateful.thresholds["ch1"] = 999.0  # Reset threshold to force non-exceedance
        
    res = engine_stateful.predict_snapshot(sample_snap)
    count = res['state_consecutive_counts']['ch1']
    sustained = res['channel_sustained_flags']['ch1']
    sys_alert = res['system_alert']
    
    print(f"Step {step_idx+1} ({step_name}): count = {count}, sustained_flag = {sustained}, system_alert = {sys_alert}")

print("\nDemonstrating reset_state():")
engine_stateful.reset_state()
print(f"State counters after reset_state(): {engine_stateful.consecutive_exceedances}")


=== STATEFUL k=3 PERSISTENCE DEMONSTRATION ===
Step 1 (Exceedance 1): count = 1, sustained_flag = False, system_alert = False
Step 2 (Exceedance 2): count = 2, sustained_flag = False, system_alert = False
Step 3 (Exceedance 3): count = 3, sustained_flag = True, system_alert = True


Step 4 (Exceedance 4): count = 4, sustained_flag = True, system_alert = True
Step 5 (Non-Exceedance): count = 0, sustained_flag = False, system_alert = False

Demonstrating reset_state():
State counters after reset_state(): {'ch1': 0, 'ch2': 0, 'ch3': 0, 'ch4': 0}


## 7. Logical OR System Alert Policy Demonstration

We demonstrate that system-level alerts follow the **Logical OR Policy**: a system alert is active if **ANY** channel achieves sustained exceedance ($k=3$).


In [7]:
print("=== LOGICAL OR SYSTEM ALERT POLICY DEMONSTRATION ===")

engine_or = AnomalyInferenceEngine(artifacts_dir=models_dir, model_type="iforest", stateful=True)
engine_or.reset_state()

for ch in CHANNELS:
    engine_or.thresholds[ch] = 999.0

# 1. Zero sustained channels
res_zero = engine_or.predict_snapshot(sample_snap)
print(f"1. Zero sustained channels -> System Alert: {res_zero['system_alert']}")

# 2. Trigger 3 consecutive exceedances on ch3 only
engine_or.thresholds["ch3"] = -999.0
_ = engine_or.predict_snapshot(sample_snap)
_ = engine_or.predict_snapshot(sample_snap)
res_ch3 = engine_or.predict_snapshot(sample_snap)

print(f"2. Channel 3 sustained ({res_ch3['channel_sustained_flags']['ch3']}), other channels False -> System Alert: {res_ch3['system_alert']}")


=== LOGICAL OR SYSTEM ALERT POLICY DEMONSTRATION ===
1. Zero sustained channels -> System Alert: False


2. Channel 3 sustained (True), other channels False -> System Alert: True

## 8. Robust Input Validation Demonstration

We demonstrate that `AnomalyInferenceEngine` catches invalid snapshot inputs (wrong row count, wrong channel count, 1D ambiguous arrays, NaNs, Infs, non-numeric) and raises clear exceptions.


In [8]:
print("=== INVALID INPUT HANDLING DEMONSTRATION ===")

# Test 1: Wrong row count (1000 rows instead of 20480)
try:
    engine_iforest.predict_snapshot(np.zeros((1000, 4)))
except ValueError as err:
    print(f"Test 1 Passed - Wrong row count caught: {err}")

# Test 2: Wrong channel count (3 channels instead of 4)
try:
    engine_iforest.predict_snapshot(np.zeros((20480, 3)))
except ValueError as err:
    print(f"Test 2 Passed - Wrong channel count caught: {err}")

# Test 3: 1D ambiguous array
try:
    engine_iforest.predict_snapshot(np.zeros((20480,)))
except ValueError as err:
    print(f"Test 3 Passed - 1D array caught: {err}")

# Test 4: Array containing NaN
nan_snap = np.zeros((20480, 4))
nan_snap[10, 0] = np.nan
try:
    engine_iforest.predict_snapshot(nan_snap)
except ValueError as err:
    print(f"Test 4 Passed - NaN array caught: {err}")

# Test 5: Array containing Inf
inf_snap = np.zeros((20480, 4))
inf_snap[10, 0] = np.inf
try:
    engine_iforest.predict_snapshot(inf_snap)
except ValueError as err:
    print(f"Test 5 Passed - Inf array caught: {err}")


=== INVALID INPUT HANDLING DEMONSTRATION ===
Test 1 Passed - Wrong row count caught: Expected raw_snapshot shape (20480, 4), got (1000, 4).
Test 2 Passed - Wrong channel count caught: Expected raw_snapshot shape (20480, 4), got (20480, 3).
Test 3 Passed - 1D array caught: Expected 2D raw_snapshot of shape (20480, 4), got 1D array with shape (20480,).
Test 4 Passed - NaN array caught: Input raw_snapshot contains NaN values.
Test 5 Passed - Inf array caught: Input raw_snapshot contains Infinite values.


## 9. Verification Summary & Methodological Limitations

### Verification Summary
- **Packaged Artifacts**: `iforest_pipeline_v1.joblib`, `pca_pipeline_v1.joblib`, `model_metadata.json` loaded cleanly.
- **Ordered Feature Name Alignment**: 28 time-domain features verified against metadata.
- **Numerical Reference Consistency**: Max score difference $\le 10^{-5}$ (IForest = `0.00e+00`, PCA = `1.77e-14`).
- **Stateful $k=3$ Persistence**: Verified $k=3$ transition and `reset_state()` resetting.
- **Logical OR Policy**: Verified system alert activates when ANY channel is sustained.
- **Robust Input Validation**: Confirmed 5 invalid input error cases rejected cleanly.

### Important Methodological Limitations
1. **Laboratory Dataset & Single Trajectory**: Based on NASA IMS Bearing Dataset Set 2 (bearing 1 outer-race defect). Results represent a single-case study ($n=1$) under constant speed (~2000 RPM) and load (6000 lbs).
2. **Assumed Healthy Baseline**: Models are fit strictly on snapshots 0..159 (first ~26.7 hours) assuming healthy initial operation. No per-snapshot physical health labels exist.
3. **No RUL or Failure-Time Prediction**: Alerts indicate statistical baseline deviation only, NOT remaining useful life (RUL), exact failure time, or physical defect classification.
4. **Security Notice**: `joblib` artifacts execute arbitrary Python bytecode during deserialization; never load model files from untrusted external sources.
